In [22]:
import pandas as pd
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import talib as ta
import yfinance as yf

In [23]:
data = yf.download(["GOOG","AAPL"],start="2019-01-01",period="2y",interval="1d")
GOOG = data.xs("GOOG", axis=1, level=1)
AAPL = data.xs("AAPL", axis=1, level=1)
GOOG.to_csv("./test_data/GOOG.csv")
AAPL.to_csv("./test_data/AAPL.csv")

[                       0%                       ]

[*********************100%***********************]  2 of 2 completed


In [24]:
AAPL = pd.read_csv("./test_data/AAPL.csv")
AAPL['Date'] = pd.to_datetime(AAPL['Date'])
AAPL = AAPL.set_index('Date')
GOOG = pd.read_csv("./test_data/GOOG.csv")
GOOG['Date'] = pd.to_datetime(GOOG['Date'])
GOOG = GOOG.set_index('Date')

In [25]:
AAPL.head()

,Close,High,Low,Open,Volume
Date,,,,,
2019-01-02,37.503723,37.724587,36.627401,36.784142,148158800
2019-01-03,33.768074,34.606398,33.722951,34.193172,365248800
2019-01-04,35.209614,35.278487,34.150430,34.323794,234428400
2019-01-07,35.131241,35.344980,34.649145,35.314106,219111200
2019-01-08,35.800953,36.055064,35.271361,35.518344,164101200


In [26]:
class SMA_Crossover(Strategy):

    def init(self):
        self.fast_ma = self.I(ta.SMA, self.data.Close, timeperiod=20)
        self.slow_ma = self.I(ta.SMA, self.data.Close, timeperiod=50)

    def next(self):
        if crossover(self.fast_ma, self.slow_ma):
            if not self.position:
                self.buy()
        elif crossover(self.slow_ma, self.fast_ma):
            self.position.close()


class RSI_Crossover(Strategy):
    upper_bound = 70
    lower_bound = 30

    def init(self):
        self.rsi = self.I(ta.RSI, self.data.Close, 14)

    def next(self):
        if crossover(self.rsi, self.upper_bound):
            self.position.close()
        elif crossover(self.rsi, self.lower_bound):
            if not self.position:
                self.buy()


In [27]:
sma_bt_AAPL = Backtest(AAPL, SMA_Crossover, cash=1000, finalize_trades=True)
sma_stats = sma_bt_AAPL.run()
metrics_sma = {
    "return_pct":           sma_stats["Return [%]"],
    "annualized_return_pct": sma_stats["Return (Ann.) [%]"],
    "sharpe_ratio":         sma_stats["Sharpe Ratio"],
    "sortino_ratio":        sma_stats["Sortino Ratio"],
    "calmar_ratio":         sma_stats["Calmar Ratio"],
    "max_drawdown_pct":     sma_stats["Max. Drawdown [%]"],
    "avg_drawdown_pct":     sma_stats["Avg. Drawdown [%]"],
    "volatility_ann_pct":   sma_stats["Volatility (Ann.) [%]"],
    "win_rate_pct":         sma_stats["Win Rate [%]"],
    "total_trades":         sma_stats["# Trades"],
    "best_trade_pct":       sma_stats["Best Trade [%]"],
    "worst_trade_pct":      sma_stats["Worst Trade [%]"],
    "avg_trade_pct":        sma_stats["Avg. Trade [%]"],
    "buyhold_return_pct":   sma_stats["Buy & Hold Return [%]"],
    "alpha_pct":            sma_stats["Alpha [%]"],
    "beta":                 sma_stats["Beta"],
}

pd.DataFrame.from_dict(metrics_sma, orient="index", columns=["value"]) \
  .to_csv("./test_data/AAPL_sma_expected.csv")

for x,y in metrics_sma.items():
    print(x,y)

return_pct 131.63673216614637
annualized_return_pct 52.06964050150591
sharpe_ratio 1.3250146949986892
sortino_ratio 3.0562356739654373
calmar_ratio 2.6264941308140144
max_drawdown_pct -19.824769410532916
avg_drawdown_pct -2.688145402877078
volatility_ann_pct 39.297406057490875
win_rate_pct 100.0
total_trades 3
best_trade_pct 59.89424047914251
worst_trade_pct 13.31980515857072
avg_trade_pct 33.785619235849374
buyhold_return_pct 194.49014102162877
alpha_pct 41.86998145334714
beta 0.4615491059920435


In [28]:
print(sma_stats["_trades"])

   Size  EntryBar  ExitBar  EntryPrice   ExitPrice    SL    TP         PnL  \
0    20       125      297   48.222119   63.728830  None  None  310.134232   
1    18       335      443   69.165400  110.591491  None  None  745.669636   
2    17       467      504  115.071217  130.398479  None  None  260.563453   

   Commission  ReturnPct  EntryTime   ExitTime Duration   Tag  \
0         0.0   0.321568 2019-07-02 2020-03-09 251 days  None   
1         0.0   0.598942 2020-05-01 2020-10-05 157 days  None   
2         0.0   0.133198 2020-11-06 2020-12-31  55 days  None   

   Entry_SMA(C,20)  Exit_SMA(C,20)  Entry_SMA(C,50)  Exit_SMA(C,50)  
0        46.815738       73.199724        46.404480       74.164497  
1        66.729102      109.539071        65.344297      110.894736  
2       112.945111      124.222340       112.710942      117.662916  


In [29]:
sma_bt_GOOG = Backtest(GOOG, SMA_Crossover, cash=1000, finalize_trades=True)
sma_stats = sma_bt_GOOG.run()
metrics_sma = {
    "return_pct":           sma_stats["Return [%]"],
    "annualized_return_pct": sma_stats["Return (Ann.) [%]"],
    "sharpe_ratio":         sma_stats["Sharpe Ratio"],
    "sortino_ratio":        sma_stats["Sortino Ratio"],
    "calmar_ratio":         sma_stats["Calmar Ratio"],
    "max_drawdown_pct":     sma_stats["Max. Drawdown [%]"],
    "avg_drawdown_pct":     sma_stats["Avg. Drawdown [%]"],
    "volatility_ann_pct":   sma_stats["Volatility (Ann.) [%]"],
    "win_rate_pct":         sma_stats["Win Rate [%]"],
    "total_trades":         sma_stats["# Trades"],
    "best_trade_pct":       sma_stats["Best Trade [%]"],
    "worst_trade_pct":      sma_stats["Worst Trade [%]"],
    "avg_trade_pct":        sma_stats["Avg. Trade [%]"],
    "buyhold_return_pct":   sma_stats["Buy & Hold Return [%]"],
    "alpha_pct":            sma_stats["Alpha [%]"],
    "beta":                 sma_stats["Beta"],
}

pd.DataFrame.from_dict(metrics_sma, orient="index", columns=["value"]) \
  .to_csv("./test_data/GOOG_sma_expected.csv")

for x,y in metrics_sma.items():
    print(x,y)

return_pct 25.624623269147012
annualized_return_pct 12.057077674119277
sharpe_ratio 0.5214846393009677
sortino_ratio 0.8492630621592099
calmar_ratio 0.6088330473599064
max_drawdown_pct -19.80358610033177
avg_drawdown_pct -3.4173024081706105
volatility_ann_pct 23.120676555845208
win_rate_pct 100.0
total_trades 3
best_trade_pct 10.680471255364466
worst_trade_pct 3.786240495041504
avg_trade_pct 8.22759791174279
buyhold_return_pct 47.76937597910296
alpha_pct 5.918089426790427
beta 0.41253488115434755


In [30]:
rsi_bt_AAPL = Backtest(AAPL, RSI_Crossover, cash=1000, finalize_trades=True)
rsi_stats = rsi_bt_AAPL.run()
metrics_rsi = {
    "return_pct":           rsi_stats["Return [%]"],
    "annualized_return_pct": rsi_stats["Return (Ann.) [%]"],
    "sharpe_ratio":         rsi_stats["Sharpe Ratio"],
    "sortino_ratio":        rsi_stats["Sortino Ratio"],
    "calmar_ratio":         rsi_stats["Calmar Ratio"],
    "max_drawdown_pct":     rsi_stats["Max. Drawdown [%]"],
    "avg_drawdown_pct":     rsi_stats["Avg. Drawdown [%]"],
    "volatility_ann_pct":   rsi_stats["Volatility (Ann.) [%]"],
    "win_rate_pct":         rsi_stats["Win Rate [%]"],
    "total_trades":         rsi_stats["# Trades"],
    "best_trade_pct":       rsi_stats["Best Trade [%]"],
    "worst_trade_pct":      rsi_stats["Worst Trade [%]"],
    "avg_trade_pct":        rsi_stats["Avg. Trade [%]"],
    "buyhold_return_pct":   rsi_stats["Buy & Hold Return [%]"],
    "alpha_pct":            rsi_stats["Alpha [%]"],
    "beta":                 rsi_stats["Beta"],
}

pd.DataFrame.from_dict(metrics_rsi, orient="index", columns=["value"]) \
  .to_csv("./test_data/AAPL_rsi_expected.csv")

for x,y in metrics_rsi.items():
    print(x,y)

return_pct 41.72842142246427
annualized_return_pct 19.008646956649322
sharpe_ratio 0.6496761515281736
sortino_ratio 1.1493103583446
calmar_ratio 0.7612134960656247
max_drawdown_pct -24.971505438220156
avg_drawdown_pct -2.7464535609960907
volatility_ann_pct 29.258649731773325
win_rate_pct 100.0
total_trades 2
best_trade_pct 31.363324445048768
worst_trade_pct 9.047357780660214
avg_trade_pct 19.686354443671572
buyhold_return_pct 253.0322924826196
alpha_pct -66.02007082140557
beta 0.42582901647334565


In [31]:
rsi_bt_GOOG = Backtest(GOOG, RSI_Crossover, cash=1000, finalize_trades=True)
rsi_stats = rsi_bt_GOOG.run()
metrics_rsi = {
    "return_pct":           rsi_stats["Return [%]"],
    "annualized_return_pct": rsi_stats["Return (Ann.) [%]"],
    "sharpe_ratio":         rsi_stats["Sharpe Ratio"],
    "sortino_ratio":        rsi_stats["Sortino Ratio"],
    "calmar_ratio":         rsi_stats["Calmar Ratio"],
    "max_drawdown_pct":     rsi_stats["Max. Drawdown [%]"],
    "avg_drawdown_pct":     rsi_stats["Avg. Drawdown [%]"],
    "volatility_ann_pct":   rsi_stats["Volatility (Ann.) [%]"],
    "win_rate_pct":         rsi_stats["Win Rate [%]"],
    "total_trades":         rsi_stats["# Trades"],
    "best_trade_pct":       rsi_stats["Best Trade [%]"],
    "worst_trade_pct":      rsi_stats["Worst Trade [%]"],
    "avg_trade_pct":        rsi_stats["Avg. Trade [%]"],
    "buyhold_return_pct":   rsi_stats["Buy & Hold Return [%]"],
    "alpha_pct":            rsi_stats["Alpha [%]"],
    "beta":                 rsi_stats["Beta"],
}

pd.DataFrame.from_dict(metrics_rsi, orient="index", columns=["value"]) \
  .to_csv("./test_data/GOOG_rsi_expected.csv")

for x,y in metrics_rsi.items():
    print(x,y)

return_pct 40.47001225287636
annualized_return_pct 18.480175210508843
sharpe_ratio 0.6440229934291523
sortino_ratio 1.1589067750786357
calmar_ratio 0.779018294148638
max_drawdown_pct -23.722389254934228
avg_drawdown_pct -3.070205171957229
volatility_ann_pct 28.694899714852824
win_rate_pct 100.0
total_trades 2
best_trade_pct 22.34891203059497
worst_trade_pct 15.663865174540014
avg_trade_pct 18.959438740094914
buyhold_return_pct 62.87920835951708
alpha_pct 5.155961106049169
beta 0.5616172987566284


In [32]:
print(AAPL.iloc[125])
print("Entry price from trades:", 48.222119)

Close     4.853815e+01
High      4.863393e+01
Low       4.821015e+01
Open      4.822212e+01
Volume    6.774080e+07
Name: 2019-07-02 00:00:00, dtype: float64
Entry price from trades: 48.222119


In [33]:
print(AAPL.iloc[106:126]['Close'].mean())
print(AAPL.iloc[106:126]['Close'].values)

46.815737915039065
[43.70419693 44.34585571 45.52621078 46.10800934 46.641922   46.49346924
 46.48389816 46.1463089  46.42163849 47.51341248 47.37454224 47.75523376
 47.5924263  47.54452896 46.82388687 47.8366394  47.82226944 47.38652802
 48.25562668 48.5381546 ]


In [34]:
import talib
sma20 = talib.SMA(AAPL['Close'].values, timeperiod=20)
print(f"SMA20 at bar 50: {sma20[50]}")
print(f"SMA20 at bar 125: {sma20[125]}")

SMA20 at bar 50: 41.87703227996826
SMA20 at bar 125: 46.815737915039065


In [35]:
print(AAPL.iloc[0], AAPL.iloc[-1])

Close     3.750372e+01
High      3.772459e+01
Low       3.662740e+01
Open      3.678414e+01
Volume    1.481588e+08
Name: 2019-01-02 00:00:00, dtype: float64 Close     1.290466e+02
High      1.310404e+02
Low       1.281033e+02
Open      1.303985e+02
Volume    9.911660e+07
Name: 2020-12-31 00:00:00, dtype: float64


In [36]:
print(sma_stats["_trades"][["Size", "EntryPrice", "EntryBar"]])

   Size  EntryPrice  EntryBar
0    17   56.622139       136
1    16   65.884094       335
2    14   82.924690       462


In [37]:
print(sma_stats["_trades"][["Size", "EntryPrice", "EntryBar"]])

   Size  EntryPrice  EntryBar
0    17   56.622139       136
1    16   65.884094       335
2    14   82.924690       462
